# FHIR at Will — functionality test

This notebook tests the two user-facing workflows only:

1. validate a known-good FHIR R4 resource with `POST /v1/validate`;
2. convert a synthetic clinical narrative with `POST /v1/NAR2FHIR`, then confirm the API returned a generated Bundle and its validation report.

## Before running

- Install the notebook environment with `uv sync --group notebook`.
- Store `FHIRBRIDGE_API_KEY` and `OPENROUTER_API_KEY` in the repository's git-ignored `.env`; the setup cell loads them automatically. The FHIR API key must include `conversions:write`.
- Optionally set `FHIRBRIDGE_BASE` and `FHIRBRIDGE_LLM_MODEL`. The API defaults to the local Compose service. Provider availability, pricing, and rate limits can change.

> Use synthetic data only. Notebook inputs and outputs are persisted in the `.ipynb` file. Never place credentials directly in a notebook cell.

In [ ]:
import json
import os
from pathlib import Path

import httpx


def load_local_env() -> None:
    """Load the repository's git-ignored .env without overwriting shell variables."""
    candidates = (Path.cwd() / ".env", Path.cwd().parent / ".env")
    env_path = next((path for path in candidates if path.is_file()), None)
    if env_path is None:
        raise FileNotFoundError("No .env found in the current or parent directory")
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = line.split("=", 1)
        os.environ.setdefault(name.strip(), value.strip())


load_local_env()

BASE_URL = os.environ.get("FHIRBRIDGE_BASE", "http://localhost:8000").rstrip("/")
API_KEY = os.environ["FHIRBRIDGE_API_KEY"]
OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
LLM_MODEL = os.environ.get("FHIRBRIDGE_LLM_MODEL", "openai/gpt-4.1-nano")

# Cold validator starts and provider models can both be slow.
client = httpx.Client(base_url=BASE_URL, timeout=300.0)
auth = {"Authorization": f"Bearer {API_KEY}"}

print(f"target: {BASE_URL}")
print(f"model:  {LLM_MODEL}")

In [ ]:
from IPython.display import HTML, display

CHECKS: list[tuple[str, bool, str]] = []


def check(label: str, passed: object, detail: str = "") -> None:
    CHECKS.append((label, bool(passed), detail))
    print(f"{'PASS' if passed else 'FAIL'}  {label}" + (f" — {detail}" if detail else ""))


def response_json(response: httpx.Response) -> dict:
    print(f"HTTP {response.status_code}  trace={response.headers.get('X-Trace-Id', '-')}")
    try:
        body = response.json()
    except ValueError:
        print(response.text[:2000])
        raise
    if response.status_code >= 400:
        print(json.dumps(body, indent=2)[:3000])
    return body


def show_report(report: dict) -> None:
    print(
        f"status={report['status']}  conformant={report['conformant']}  "
        f"resource={report['resource_type']}  duration={report['duration_ms']} ms"
    )
    rows = []
    for layer in report["layers"]:
        issues = "<br>".join(
            f"<code>{issue['severity']}</code> {issue['message'][:180]}"
            for issue in layer["issues"][:3]
        )
        rows.append(
            f"<tr><td>L{layer['layer_number']}</td><td>{layer['layer']}</td>"
            f"<td>{layer['status']}</td><td>{layer['errors']}</td>"
            f"<td>{layer['warnings']}</td><td>{issues}</td></tr>"
        )
    display(
        HTML(
            "<table><thead><tr><th>#</th><th>layer</th><th>status</th>"
            "<th>errors</th><th>warnings</th><th>issues</th></tr></thead>"
            f"<tbody>{''.join(rows)}</tbody></table>"
        )
    )

## 1. Validate a FHIR resource

This sends a known-good synthetic heart-rate Observation through `POST /v1/validate`.
The test confirms that the request succeeds, the resource is conformant, all eight
report layers are present, and no layer fails.

In [ ]:
VALID_OBSERVATION = {
    "resourceType": "Observation",
    "text": {
        "status": "generated",
        "div": '<div xmlns="http://www.w3.org/1999/xhtml">Heart rate 72/min</div>',
    },
    "status": "final",
    "category": [
        {
            "coding": [
                {
                    "system": "http://terminology.hl7.org/CodeSystem/observation-category",
                    "code": "vital-signs",
                    "display": "Vital Signs",
                }
            ]
        }
    ],
    "code": {
        "coding": [
            {
                "system": "http://loinc.org",
                "code": "8867-4",
                "display": "Heart rate",
            }
        ]
    },
    "subject": {"reference": "Patient/example"},
    "performer": [{"reference": "Practitioner/example"}],
    "effectiveDateTime": "2024-01-15T09:30:00Z",
    "valueQuantity": {
        "value": 72,
        "unit": "/min",
        "system": "http://unitsofmeasure.org",
        "code": "/min",
    },
}

validation_response = client.post(
    "/v1/validate",
    headers=auth,
    json={"resource": VALID_OBSERVATION},
)
validation_report = response_json(validation_response)

check("validation request succeeds", validation_response.status_code == 200)
if validation_response.status_code == 200:
    show_report(validation_report)
    check("resource is conformant", validation_report["conformant"])
    check("routing decision is auto", validation_report["status"] == "auto")
    check("report contains all eight layers", len(validation_report["layers"]) == 8)
    check(
        "no validation layer failed",
        not any(layer["status"] == "failed" for layer in validation_report["layers"]),
    )

## 2. Convert narrative to FHIR

This sends a synthetic clinical note to `POST /v1/NAR2FHIR` using your OpenRouter key.
The endpoint performs grounded extraction, generates a FHIR Bundle, and validates the
result.

A generated Bundle is not required to be conformant for this functionality test because
model quality varies. The test passes when the request succeeds and the response contains
a FHIR Bundle, a complete validation report, and LLM call metadata.

In [ ]:
SYNTHETIC_NOTE = (
    "62-year-old male seen for routine follow-up on 2024-01-15. "
    "Blood pressure 128/82 mmHg and heart rate 74/min. "
    "History of type 2 diabetes mellitus. "
    "Currently takes metformin 500 mg by mouth twice daily."
)

conversion_headers = {
    **auth,
    "X-LLM-Provider": "openrouter",
    "X-LLM-Model": LLM_MODEL,
    "X-LLM-API-Key": OPENROUTER_API_KEY,
    "X-PHI-Egress-Acknowledged": "true",
}
conversion_response = client.post(
    "/v1/NAR2FHIR",
    headers=conversion_headers,
    json={"text": SYNTHETIC_NOTE},
)
conversion = response_json(conversion_response)

check("NAR2FHIR request succeeds", conversion_response.status_code == 200)
if conversion_response.status_code == 200:
    bundle = conversion["bundle"]
    report = conversion["report"]
    llm = conversion["llm"]

    check("conversion returned a FHIR Bundle", bundle.get("resourceType") == "Bundle")
    check("generated Bundle contains resources", bool(bundle.get("entry")))
    check("generated Bundle was validated", len(report.get("layers", [])) == 8)
    check("response includes model metadata", bool(llm.get("model")))
    check("conversion id is present", bool(conversion.get("conversion_id")))

    print(
        f"model={llm['model']}  tokens={llm.get('usage', {}).get('total_tokens', 'unknown')}  "
        f"latency={llm.get('latency_ms', 0)} ms  cost={llm.get('cost_usd')}"
    )

    show_report(report)
    display(HTML("<h4>Generated Bundle</h4>"))
    print(json.dumps(bundle, indent=2)[:8000])

## Results

The notebook succeeds when the validation and conversion workflows both complete and
return the expected response structures.

In [ ]:
passed = sum(1 for _, ok, _ in CHECKS if ok)
failed = [(label, detail) for label, ok, detail in CHECKS if not ok]

color = "#137333" if not failed else "#c5221f"
display(HTML(f"<h3 style='color:{color}'>{passed}/{len(CHECKS)} checks passed</h3>"))

if failed:
    print("\nFailing checks:")
    for label, detail in failed:
        print(f"- {label}" + (f": {detail}" if detail else ""))
else:
    print("\nValidation and NAR2FHIR conversion are working.")

client.close()